# PE/N-drop populations using the v3 data layout

This analysis compares the PE/N-drop distributions for states classified as phase-separated and non-phase-separated by the v3 voxel classifier. It reads the canonical v3 metadata and computes PE-drop statistics in memory without modifying simulation logs.

In [ ]:
# ============================================================
# Compare v3 PE/N drop against v3 voxel phase separation
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from md_Helpers import classification
from md_Helpers import metadata
from md_Helpers.paths import THERMALIZED_STATES_V3_ROOT


# ============================================================
# Settings
# ============================================================

base_folder = Path(THERMALIZED_STATES_V3_ROOT)
phase_name = "randomization"
n_hist_bins = 40

# When voxel metadata is absent, classify the saved final state in
# memory. dry_run=True ensures that the HDF5 log is not modified.
compute_missing_voxel = True


# ============================================================
# Read the canonical v3 voxel result
# ============================================================

def get_voxel_result(log_path):
    voxel_result, metadata_path = (
        classification.read_phase_method_attrs(
            log_path,
            "voxel",
        )
    )

    if "phase_separated" in voxel_result:
        return voxel_result, metadata_path, "saved_metadata"

    if not compute_missing_voxel:
        return {}, metadata_path, "missing"

    voxel_result = (
        classification.write_voxel_phase_separation_metadata(
            log_path=log_path,
            dry_run=True,
        )
    )

    return voxel_result, metadata_path, "computed_in_memory"


# ============================================================
# Collect v3 data
# ============================================================

log_paths = sorted(
    base_folder.glob(f"**/{phase_name}_log.hdf5")
)

rows = []
error_rows = []

for log_path in log_paths:
    try:
        state_metadata = metadata.read_attrs(
            log_path,
            "metadata/state",
        )
        run_metadata = metadata.read_attrs(
            log_path,
            "metadata/run",
        )

        voxel_result, voxel_metadata_path, voxel_source = (
            get_voxel_result(log_path)
        )

        if "phase_separated" not in voxel_result:
            raise KeyError(
                "No v3 voxel phase-separation result is available"
            )

        # Compute PE-drop values from the log without writing metadata.
        pe_result = (
            classification.compute_PE_drop_phase_separation_from_log(
                log_path
            )
        )

        rows.append(
            {
                "n_fcc_cells": state_metadata.get(
                    "n_fcc_cells",
                    np.nan,
                ),
                "target_rho": state_metadata.get(
                    "target_rho",
                    np.nan,
                ),
                "actual_rho": state_metadata.get(
                    "actual_rho",
                    np.nan,
                ),
                "kT": state_metadata.get("kT", np.nan),
                "nsteps": run_metadata.get("nsteps", np.nan),
                "seed": run_metadata.get("seed", np.nan),
                "voxel_phase_separated": bool(
                    voxel_result["phase_separated"]
                ),
                "voxel_low_density_fraction": voxel_result.get(
                    "low_density_fraction",
                    np.nan,
                ),
                "voxel_result_source": voxel_source,
                "voxel_metadata_path": voxel_metadata_path,
                "PE_drop_phase_separated": bool(
                    pe_result["phase_separated"]
                ),
                "PE_drop": pe_result["PE_drop"],
                "PE_drop_z_score": pe_result[
                    "PE_drop_z_score"
                ],
                "starting_PE_per_particle": pe_result[
                    "starting_PE_per_particle"
                ],
                "last_PE_per_particle_mean": pe_result[
                    "last_PE_per_particle_mean"
                ],
                "last_PE_per_particle_std": pe_result[
                    "last_PE_per_particle_std"
                ],
                "log_path": str(log_path),
            }
        )

    except Exception as error:
        error_rows.append(
            {
                "log_path": str(log_path),
                "error": repr(error),
            }
        )

df_compare = pd.DataFrame(rows)
df_errors = pd.DataFrame(error_rows)

print("V3 root:", base_folder)
print("Logs found:", len(log_paths))
print("Usable rows:", len(df_compare))
print("Skipped rows:", len(df_errors))

if df_compare.empty:
    if not df_errors.empty:
        display(df_errors.head(20))

    raise RuntimeError(
        "No usable v3 rows were found. Check the displayed errors and "
        "confirm that THERMALIZED_STATES_V3_ROOT points to the data."
    )


# ============================================================
# Population counts and summary statistics
# ============================================================

phase_order = [False, True]
phase_labels = ["Not phase-separated", "Phase-separated"]

population_counts = (
    df_compare["voxel_phase_separated"]
    .value_counts()
    .reindex(phase_order, fill_value=0)
    .rename("count")
)

population_table = population_counts.to_frame()
population_table["fraction"] = (
    population_table["count"] / len(df_compare)
)
population_table.index = phase_labels

print("\nVoxel-classified populations:")
display(population_table)

count_difference = int(
    population_counts.loc[False]
    - population_counts.loc[True]
)

print(
    "Population difference "
    "(not phase-separated minus phase-separated):",
    count_difference,
)

summary = (
    df_compare
    .groupby("voxel_phase_separated")[
        ["PE_drop", "PE_drop_z_score"]
    ]
    .agg(["count", "mean", "std", "median", "min", "max"])
)

print("\nPE-drop summary by voxel population:")
display(summary)

print("\nVoxel versus PE-drop classifier counts:")
display(
    pd.crosstab(
        df_compare["voxel_phase_separated"],
        df_compare["PE_drop_phase_separated"],
        rownames=["Voxel classifier"],
        colnames=["PE-drop classifier"],
        margins=True,
    )
)


# ============================================================
# Plot population size and PE-drop distributions
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {False: "tab:blue", True: "tab:orange"}

bars = axes[0].bar(
    phase_labels,
    population_counts.to_numpy(),
    color=[colors[False], colors[True]],
)
axes[0].bar_label(bars, padding=3)
axes[0].set_ylabel("Number of states")
axes[0].set_title("Voxel-classified population sizes")
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", rotation=15)

plot_specs = [
    (
        "PE_drop",
        classification.DEFAULT_PHASE_SEP_PE_DROP_THRESHOLD,
        "PE/N drop",
    ),
    (
        "PE_drop_z_score",
        -classification.DEFAULT_PHASE_SEP_PE_DROP_Z_LIMIT,
        "PE/N-drop z score",
    ),
]

for ax, (quantity, threshold, xlabel) in zip(axes[1:], plot_specs):
    finite_all = df_compare.loc[
        np.isfinite(df_compare[quantity]),
        quantity,
    ].to_numpy(dtype=float)

    if finite_all.size == 0:
        ax.text(
            0.5,
            0.5,
            f"No finite {quantity} values",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        continue

    bin_edges = np.histogram_bin_edges(
        finite_all,
        bins=n_hist_bins,
    )

    for phase_value, label in zip(phase_order, phase_labels):
        values = df_compare.loc[
            (df_compare["voxel_phase_separated"] == phase_value)
            & np.isfinite(df_compare[quantity]),
            quantity,
        ].to_numpy(dtype=float)

        histogram, _ = np.histogram(values, bins=bin_edges)

        ax.stairs(
            histogram,
            bin_edges,
            linewidth=2,
            color=colors[phase_value],
            label=f"{label} (N={len(values)})",
        )

    ax.axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"PE-drop threshold = {threshold:g}",
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of states")
    ax.set_title(f"{xlabel} population comparison")
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(
    "V3 PE/N-drop populations by voxel phase separation",
    fontsize=16,
)
fig.tight_layout()
plt.show()


In [ ]:
# Inspect the rows used in the plots.
df_compare.sort_values(
    ["voxel_phase_separated", "n_fcc_cells", "kT", "target_rho"]
).reset_index(drop=True)

## States where the two phase-separation methods disagree

In [ ]:
# Keep only states assigned different labels by the voxel and PE-drop methods.
disagreement_mask = (
    df_compare["voxel_phase_separated"]
    != df_compare["PE_drop_phase_separated"]
)

df_disagreements = df_compare.loc[disagreement_mask].copy()

df_disagreements["disagreement_type"] = np.where(
    df_disagreements["voxel_phase_separated"],
    "Voxel=True, PE-drop=False",
    "Voxel=False, PE-drop=True",
)

df_disagreements = df_disagreements.sort_values(
    [
        "disagreement_type",
        "n_fcc_cells",
        "kT",
        "target_rho",
    ]
).reset_index(drop=True)

n_disagreements = len(df_disagreements)
disagreement_fraction = n_disagreements / len(df_compare)

print("Total states compared:", len(df_compare))
print("Disagreements:", n_disagreements)
print(f"Disagreement fraction: {disagreement_fraction:.2%}")

print("\nDisagreement direction counts:")
display(
    df_disagreements["disagreement_type"]
    .value_counts()
    .rename("count")
    .to_frame()
)

disagreement_columns = [
    "disagreement_type",
    "n_fcc_cells",
    "target_rho",
    "actual_rho",
    "kT",
    "nsteps",
    "seed",
    "voxel_phase_separated",
    "PE_drop_phase_separated",
    "voxel_low_density_fraction",
    "PE_drop",
    "PE_drop_z_score",
    "starting_PE_per_particle",
    "last_PE_per_particle_mean",
    "last_PE_per_particle_std",
    "log_path",
]

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    120,
):
    display(df_disagreements[disagreement_columns])


In [ ]:
# Inspect any logs that could not be analyzed.
df_errors